[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C52_Industrial_Research_Practice_Course/05_research_output/05_research_output_ip.ipynb)

# 05 · 研究产出与知识产权（可专利性自检 / 权利要求结构 / 许可兼容 / 演示模板）

> ⚠️ **本 notebook 是工程视角的实用工具，不是法律意见。** 真实的专利申请、许可合规判断
> 必须由法务/IP 部门或执业律师做。这里的目标是让你**知道该在什么时候找他们、带什么材料去**。

目标：把 **可专利性三门槛 → 权利要求宽窄 → 发明披露书 → 许可兼容矩阵 → 演示结构检查**
做成可运行的检查器。

路线：可专利性自检（含「公开即丧失」的硬拦截）→ 权利要求宽窄与绕过分析 →
披露书完整性检查 → 许可兼容矩阵与**AGPL 的网络触发** → 依赖树传染性传播 →
演示结构检查（结论先行 / 明确 ask）→ 产出投入产出账 → ✏️ 练习 → 📖 答案 → 🧪 模板胶囊。

> 心智模型：**研究的价值 ≠ 研究的产出。中间隔着一次「翻译成组织能消费的形式」的转换，
> 而这个转换是你的责任。**

## 1 · 可专利性自检：三个门槛 + 一个硬拦截

In [ ]:
import numpy as np, collections, itertools, json, textwrap

DISCLOSURE_KINDS = ['arxiv', 'paper', 'blog', 'open_source', 'public_talk',
                    'public_commit', 'customer_demo', 'none']

def patentability_selfcheck(invention):
    '''工程视角的自检。返回 (verdict, 逐项结论)。verdict: 'blocked'|'weak'|'promising'。'''
    notes, blocking = [], False

    # 硬拦截：已公开 -> 新颖性丧失（多数辖区不可逆）
    pub = invention.get('public_disclosure', 'none')
    if pub != 'none':
        blocking = True
        notes.append(f'❌ **硬拦截**：已通过 {pub} 公开 -> 新颖性可能已丧失。'
                     f'立刻找法务（美国有 12 个月宽限期，但不适用于所有辖区）')
    else:
        notes.append('✅ 尚未对外公开 -> 新颖性门槛未被破坏')

    # 门槛 1：新颖性（相对现有技术）
    if invention.get('closest_prior_art_delta', '') in ('', 'none'):
        notes.append('🔶 新颖性：说不清与最接近现有技术的差别 -> 先做检索')
    else:
        notes.append(f'✅ 新颖性：与现有技术的差别 = {invention["closest_prior_art_delta"]}')

    # 门槛 2：非显而易见性 —— **「非预期的效果」是最有力的论据**
    if invention.get('unexpected_effect'):
        notes.append(f'✅ 非显而易见：有非预期效果「{invention["unexpected_effect"]}」（强论据）')
    elif invention.get('is_transfer_only'):
        notes.append('🔶 非显而易见：只是「把 A 方法用到 B 任务」-> 常被认为显而易见')
    else:
        notes.append('🔶 非显而易见：只有「按预期地好一点」-> 论据偏弱')

    # 门槛 3：可专利主题 —— **算法 + 具体技术效果**
    tech = invention.get('technical_effect')
    if tech:
        notes.append(f'✅ 可专利主题：绑定了具体技术效果「{tech}」')
    else:
        notes.append('❌ 可专利主题：纯算法/数学，未绑定技术效果 -> 很难通过')

    strong = sum(1 for n in notes if n.startswith('✅'))
    if blocking:
        v = 'blocked'
    elif strong >= 3 and tech:
        v = 'promising'
    else:
        v = 'weak'
    return v, notes

INVENTIONS = [
    ('已发 arXiv 的好想法', dict(
        public_disclosure='arxiv', closest_prior_art_delta='动态权重调度',
        unexpected_effect='参数更少反而更鲁棒', technical_effect='推理显存降低 40%')),
    ('纯损失函数改进', dict(
        public_disclosure='none', closest_prior_art_delta='新的正则项',
        technical_effect=None)),
    ('A 方法搬到 B 任务', dict(
        public_disclosure='none', closest_prior_art_delta='应用领域不同',
        is_transfer_only=True, technical_effect='端到端延迟降低 15%')),
    ('绑定技术效果 + 非预期收益', dict(
        public_disclosure='none', closest_prior_art_delta='训练中动态调整损失权重',
        unexpected_effect='减少参数反而提升鲁棒性',
        technical_effect='推理显存降低 40%，无需改动推理代码')),
]
for name, inv in INVENTIONS:
    v, notes = patentability_selfcheck(inv)
    print(f'=== {name} -> **{v}** ===')
    for n in notes: print('   ' + n)
    print()

assert patentability_selfcheck(INVENTIONS[0][1])[0] == 'blocked', '已公开 -> 硬拦截'
assert patentability_selfcheck(INVENTIONS[1][1])[0] == 'weak', '纯算法无技术效果 -> 弱'
assert patentability_selfcheck(INVENTIONS[3][1])[0] == 'promising'
print('⚠️  **「先发 arXiv 占坑再申请专利」在多数辖区不可行** ——')
print('    正确顺序：有想法 -> 记录（带日期）-> 内部披露 -> 法务确认 -> 再决定公开。')
print('✅ 把「算法创新」翻译成「技术问题 + 技术手段 + **技术效果**」三段式，是关键动作。')

## 2 · 权利要求的宽窄：绕过分析

**「如果竞争者要绕过我的专利，最小的改动是什么？」** 花十分钟，能显著提升专利质量。

In [ ]:
class Claim:
    def __init__(self, limitations):
        self.limitations = list(limitations)     # 每条限定
    def breadth(self):
        '''限定越多，保护范围越窄。'''
        return 1.0 / (1 + len(self.limitations))
    def covers(self, implementation):
        '''实现必须满足**所有**限定才落入保护范围。'''
        return all(any(lim in feat for feat in implementation) for lim in self.limitations)
    def __repr__(self):
        return '一种方法，包括：\n' + '\n'.join(
            f'  {chr(97+i)}) {l}' for i, l in enumerate(self.limitations))

# 过窄的权利要求：把实现细节都写进去了
narrow = Claim(['使用 ReLU 激活', '在 Transformer 的第 12 层', '使用 fp16 精度',
                '动态调整损失权重'])
# 合理宽度：只保留发明的核心机制
broad = Claim(['在训练过程中根据中间层统计量动态调整损失权重'])

IMPLS = {
    '我的实现':        ['使用 ReLU 激活', '在 Transformer 的第 12 层', '使用 fp16 精度',
                        '在训练过程中根据中间层统计量动态调整损失权重', '动态调整损失权重'],
    '竞争者：换 GELU': ['使用 GELU 激活', '在 Transformer 的第 12 层', '使用 fp16 精度',
                        '在训练过程中根据中间层统计量动态调整损失权重', '动态调整损失权重'],
    '竞争者：换 bf16': ['使用 ReLU 激活', '在 Transformer 的第 12 层', '使用 bf16 精度',
                        '在训练过程中根据中间层统计量动态调整损失权重', '动态调整损失权重'],
    '竞争者：第 8 层': ['使用 ReLU 激活', '在 Transformer 的第 8 层', '使用 fp16 精度',
                        '在训练过程中根据中间层统计量动态调整损失权重', '动态调整损失权重'],
    '真正不同的方案':  ['使用固定的损失权重', '在 Transformer 的第 12 层'],
}
print(f"{'实现':<22s} {'过窄的权利要求':>16s} {'合理宽度':>12s}")
for name, impl in IMPLS.items():
    print(f'{name:<22s} {("落入 ❌被覆盖" if narrow.covers(impl) else "绕过 ✅"):>18s} '
          f'{("落入" if broad.covers(impl) else "绕过"):>12s}')

assert narrow.covers(IMPLS['我的实现'])
assert not narrow.covers(IMPLS['竞争者：换 GELU']), '换个激活函数就绕过了过窄的权利要求'
assert broad.covers(IMPLS['竞争者：换 GELU']), '合理宽度仍覆盖'
assert not broad.covers(IMPLS['真正不同的方案']), '不该覆盖真正不同的方案'
assert broad.breadth() > narrow.breadth()
print(f'\n保护范围: 过窄 {narrow.breadth():.2f} vs 合理 {broad.breadth():.2f}')
print('\n⚠️  过窄的权利要求：竞争者「把 ReLU 换成 GELU」就绕过了。')
print('✅ 你（发明人）的独特贡献是**列出所有能想到的变体** ——')
print('   不同架构、不同精度、不同粒度、不同层。让代理人决定哪些进权利要求。')

VARIANTS = ['任意激活函数（ReLU/GELU/SiLU）', '任意层（不限第 12 层）',
            '任意精度（fp32/fp16/bf16/int8）', '任意统计量（均值/方差/范数/梯度范数）',
            '也适用于 CNN 与 RNN，不限 Transformer',
            '权重调整可以是连续的或离散分档的']
print(f'\n变体清单（披露书第 ⑤ 节，最容易被跳过也最有价值）：')
for v in VARIANTS: print(f'  · {v}')
assert len(VARIANTS) >= 5

## 3 · 发明披露书的完整性检查

In [ ]:
REQUIRED_SECTIONS = {
    'problem':        '① 现有技术解决不了什么**技术**问题',
    'prior_art':      '② 最接近的已有方案，以及为什么不够（诚实列出）',
    'solution':       '③ 核心机制（配图/伪代码，越具体越好）',
    'why_works':      '④ 机制解释 + 实验证据（标注非预期效果）',
    'variants':       '⑤ 所有能想到的替代实现（**最易跳过、最有价值**）',
    'timeline':       '⑥ 构思日期 / 是否对外提过 / 代码在哪（**决定还能不能申请**）',
}

def check_disclosure(doc):
    missing = [k for k in REQUIRED_SECTIONS if not doc.get(k)]
    warnings = []
    if doc.get('variants') and len(doc['variants']) < 3:
        warnings.append('⑤ 变体少于 3 条 -> 权利要求可能过窄')
    tl = doc.get('timeline') or {}
    if tl.get('public_disclosure', 'none') != 'none':
        warnings.append(f'⑥ 已通过 {tl["public_disclosure"]} 公开 -> **立刻找法务**')
    if not tl.get('conception_date'):
        warnings.append('⑥ 缺构思日期 -> 优先权与发明人认定会有麻烦')
    return {'missing': missing, 'warnings': warnings,
            'ok': not missing and not warnings}

draft = {
    'problem': '长序列训练时激活显存随序列长平方增长，限制了可用的上下文长度',
    'prior_art': 'FlashAttention 解决注意力矩阵，但 FFN 中间激活仍是瓶颈；'
                 '梯度检查点全局重算，代价 30% 计算',
    'solution': '按中间层统计量动态选择哪些层重算，形成自适应的重算策略',
    'why_works': '显存-计算的帕累托前沿更优；**非预期效果**：重算的层数减少后收敛更快',
    'variants': VARIANTS,
    'timeline': {'conception_date': '2026-03-14', 'public_disclosure': 'none',
                 'code': 'internal-git/exp/adaptive-ckpt'},
}
r = check_disclosure(draft)
print('完整披露书检查:', r)
assert r['ok'], r

incomplete = {k: v for k, v in draft.items() if k not in ('variants', 'why_works')}
r2 = check_disclosure(incomplete)
print('\n缺 ④⑤ 的草稿:')
for k in r2['missing']: print(f'   缺 {REQUIRED_SECTIONS[k]}')
assert set(r2['missing']) == {'variants', 'why_works'}

leaked = dict(draft); leaked['timeline'] = dict(draft['timeline'],
                                                public_disclosure='public_talk')
r3 = check_disclosure(leaked)
assert any('立刻找法务' in w for w in r3['warnings'])
print(f'\n已公开的草稿: {r3["warnings"]}')
print('\n⚠️  即使是「内部但有外部人参加的会议」、「公开 Slack 频道」、「公开仓库 commit」、')
print('    「给客户做 demo」都可能构成公开。**先记录、先披露，再决定公开与否。**')

## 4 · 许可兼容矩阵：日常最容易踩的坑

In [ ]:
LICENSES = {
    'MIT':          dict(kind='permissive', copyleft=0, network_trigger=False, commercial=True),
    'BSD-3-Clause': dict(kind='permissive', copyleft=0, network_trigger=False, commercial=True),
    'Apache-2.0':   dict(kind='permissive', copyleft=0, network_trigger=False, commercial=True),
    'LGPL-3.0':     dict(kind='weak-copyleft', copyleft=1, network_trigger=False, commercial=True),
    'GPL-2.0':      dict(kind='strong-copyleft', copyleft=2, network_trigger=False, commercial=True),
    'GPL-3.0':      dict(kind='strong-copyleft', copyleft=2, network_trigger=False, commercial=True),
    'AGPL-3.0':     dict(kind='network-copyleft', copyleft=3, network_trigger=True, commercial=True),
    'CC-BY-NC-4.0': dict(kind='non-commercial', copyleft=0, network_trigger=False, commercial=False),
    'Llama-Community': dict(kind='custom', copyleft=0, network_trigger=False, commercial='conditional'),
}

def can_use(license_name, product_closed_source, distributed, network_service,
            commercial_use=True):
    '''工程视角的粗筛。真实判断交法务。'''
    L = LICENSES[license_name]
    if commercial_use and L['commercial'] is False:
        return False, f'{license_name} 禁止商业使用'
    if L['commercial'] == 'conditional':
        return 'review', f'{license_name} 是自定义许可（用户数阈值/可接受使用政策）-> **逐条读**'
    if L['copyleft'] == 0:
        return True, f'{license_name} 宽松许可，闭源产品可用（保留声明/标注修改）'
    if not product_closed_source:
        return True, f'{license_name}：你的产品本身开源 -> 兼容'
    # 闭源产品 + copyleft
    if L['network_trigger'] and network_service:
        return False, (f'{license_name}：**「通过网络提供服务」即触发开源义务** '
                       f'-> 闭源 SaaS 不可用（最容易踩的坑）')
    if L['copyleft'] >= 2 and distributed:
        return False, f'{license_name}：分发衍生作品必须同样开源 -> 闭源产品不可用'
    if L['copyleft'] == 1:
        return 'review', f'{license_name}：动态链接通常可以，静态链接有争议 -> 交法务'
    if L['copyleft'] >= 2 and not distributed and not network_service:
        return True, f'{license_name}：仅内部使用、不分发 -> 义务未触发（但别以此为长期策略）'
    return 'review', f'{license_name}：边界情形 -> 交法务'

SCENARIOS = [
    ('闭源 SaaS 推理服务', True, False, True),
    ('闭源本地分发的 App', True, True, False),
    ('仅内部使用的工具',   True, False, False),
    ('我们自己也开源',     False, True, True),
]
for sc, closed, dist, net in SCENARIOS:
    print(f'=== {sc} ===')
    for lic in ['MIT', 'Apache-2.0', 'LGPL-3.0', 'GPL-3.0', 'AGPL-3.0',
                'CC-BY-NC-4.0', 'Llama-Community']:
        ok, why = can_use(lic, closed, dist, net)
        mark = {True: '✅', False: '❌', 'review': '🔶'}[ok]
        print(f'  {mark} {lic:<18s} {why}')
    print()

assert can_use('AGPL-3.0', True, False, True)[0] is False, 'AGPL + 闭源 SaaS -> 不行'
assert can_use('GPL-3.0', True, False, False)[0] is True, 'GPL + 仅内部使用 -> 义务未触发'
assert can_use('GPL-3.0', True, True, False)[0] is False, 'GPL + 分发 -> 不行'
assert can_use('CC-BY-NC-4.0', True, False, False)[0] is False, 'NC 禁止商业使用'
assert can_use('MIT', True, True, True)[0] is True
print('⚠️  **AGPL 打破了「不分发就没有义务」这个直觉** ——')
print('    GPL 的义务在分发时触发（内部使用安全）；')
print('    AGPL 的义务在「通过网络提供功能」时就触发 —— 这正是所有 AI 服务的形态。')
print('⚠️  **大量数据集是 CC-BY-NC** —— 训练商业模型即违约。')

In [ ]:
# copyleft 的传染性：沿依赖树传播
DEPS = {
    'my-service':   ['inference-lib', 'vector-db', 'utils'],
    'inference-lib':['onnxruntime', 'numpy'],
    'vector-db':    ['some-agpl-index'],          # ← 藏在二级依赖里
    'utils':        ['requests'],
    'onnxruntime':  [], 'numpy': [], 'requests': [], 'some-agpl-index': [],
}
DEP_LICENSE = {
    'inference-lib': 'Apache-2.0', 'vector-db': 'MIT', 'utils': 'MIT',
    'onnxruntime': 'MIT', 'numpy': 'BSD-3-Clause', 'requests': 'Apache-2.0',
    'some-agpl-index': 'AGPL-3.0',                # ← 真正的问题在这里
}

def scan_licenses(root, deps, lic):
    '''遍历依赖树，返回 (全部许可, 最强 copyleft, 路径)。'''
    seen, worst, path_to_worst = set(), None, None
    def walk(node, path):
        nonlocal worst, path_to_worst
        if node in seen: return
        seen.add(node)
        L = lic.get(node)
        if L:
            cl = LICENSES[L]['copyleft']
            if worst is None or cl > LICENSES[worst]['copyleft']:
                worst, path_to_worst = L, path + [node]
        for d in deps.get(node, []):
            walk(d, path + [node])
    walk(root, [])
    return sorted({lic[n] for n in seen if n in lic}), worst, path_to_worst

all_lic, worst, path = scan_licenses('my-service', DEPS, DEP_LICENSE)
print('依赖树里的全部许可:', all_lic)
print(f'最强 copyleft: **{worst}**')
print('传播路径: ' + ' -> '.join(path))
ok, why = can_use(worst, product_closed_source=True, distributed=False, network_service=True)
print(f'\n闭源 SaaS 场景下: {"✅" if ok is True else "❌" if ok is False else "🔶"} {why}')
assert worst == 'AGPL-3.0' and ok is False
assert 'some-agpl-index' in path and 'vector-db' in path
print('\n⚠️  问题藏在**二级依赖**里：vector-db 自己是 MIT，但它依赖一个 AGPL 的索引库。')
print('✅ 两条操作规则：')
print('   ① 许可检查要在**引入依赖时**做，不是发布前做（发布前发现要重写已写好的代码）')
print('   ② **不要自己判断边界情形** —— 你的职责是准确记录「用了什么、什么许可、怎么用」')

## 5 · 演示结构检查：结论先行 + 明确 ask

In [ ]:
ACADEMIC_TEMPLATE = ['背景与动机', '相关工作', '方法', '实验设置', '结果',
                     '消融实验', '结论与未来工作']
INTERNAL_TEMPLATE = ['结论与建议', '为什么值得做（1 张）', '关键证据',
                     '代价与风险', '与现有方案的对比', '**明确的 ask**']

def check_talk_structure(slide_titles, ask=None, time_split=None):
    problems = []
    # ① 结论先行
    first = slide_titles[0] if slide_titles else ''
    if not any(k in first for k in ['结论', '建议', '要点', 'TL;DR', '我们发现']):
        problems.append(f'① 第一张是「{first}」而不是结论 -> 听众可能只听前五分钟')
    # ② 标题应是断言而不是章节名
    section_names = ['背景', '相关工作', '方法', '实验设置', '结果', '消融', '未来工作']
    generic = [t for t in slide_titles if any(t.strip() == s for s in section_names)]
    if generic:
        problems.append(f'② 标题是章节名而非断言: {generic} -> 连起来读不成论证')
    # ③ 必须有明确 ask
    if not ask:
        problems.append('③ 没有明确的 ask -> 最好的结果是「挺有意思」，然后什么都不发生')
    elif not any(k in ask for k in ['批准', '决定', '评估', '分配', '由']):
        problems.append(f'③ ask 不够具体（「{ask}」）-> 要说清谁在什么时候做什么决定')
    # ④ 时间分配
    if time_split and time_split.get('method', 0) > 0.4:
        problems.append(f'④ 方法占 {time_split["method"]:.0%} -> 内部演示应 ~20% 方法、'
                        f'80% 影响与代价')
    # ⑤ 必须讲局限
    if not any('风险' in t or '代价' in t or '局限' in t for t in slide_titles):
        problems.append('⑤ 没有代价/风险页 -> 听众无法评估风险，通常会因此不批准')
    return {'problems': problems, 'ok': not problems}

bad = check_talk_structure(ACADEMIC_TEMPLATE, ask=None,
                           time_split={'method': 0.8, 'impact': 0.2})
print('用学术模板做内部演示:')
for p_ in bad['problems']: print('   ❌ ' + p_)
assert len(bad['problems']) >= 4

good_titles = ['结论：自适应重算把显存降 40%，精度损失 0.3 分',
               '为什么值得做：解锁 32k 上下文，无需换卡',
               '关键证据：3 个模型 × 2 个数据集上稳定复现',
               '代价与风险：训练慢 8%；长序列下需重新调参',
               '与现有方案对比：优于全局梯度检查点（快 22%）',
               'ask：批准 200 GPU 小时做 70B 规模验证']
good = check_talk_structure(good_titles,
                            ask='批准 200 GPU 小时做 70B 规模验证，由平台组在两周内评估集成成本',
                            time_split={'method': 0.2, 'impact': 0.8})
print('\n用内部演示模板:')
print('   ✅ 通过' if good['ok'] else good['problems'])
assert good['ok'], good['problems']

print('\n两个立刻可用的检查:')
print('  ① 只看第一张和最后一张 —— 一个只看这两张的人能做出决定吗？')
print('  ② 把所有标题连起来读 —— 应该构成一个完整的论证。')
print('     **标题应该是断言**（「显存降 40%，精度损失 0.3 分」），不是章节名（「结果」）。')
first_last = [good_titles[0], good_titles[-1]]
assert '结论' in first_last[0] and 'ask' in first_last[1]
print(f'\n本例的第一张+最后一张:\n  {first_last[0]}\n  {first_last[1]}')
print('✅ 能据此做决定 -> 结构合格。')

## 6 · 产出形式的投入产出账

In [ ]:
OUTPUTS = [
    ('专利（一件）',      30,   'high',   'high',   '1–4 年'),
    ('顶会论文（一篇）',  300,  'medium', 'high',   '6–18 个月'),
    ('内部演示（一次）',  8,    'high',   'high',   '即时'),
    ('开源发布',          80,   'medium', 'medium', '数周–数月'),
    ('内部文档/工具',     20,   'high',   'low',    '持续'),
]
SCORE = {'high': 3, 'medium': 2, 'low': 1}

print(f"{'产出':<20s} {'工时':>6s} {'公司价值':>9s} {'个人价值':>9s} "
      f"{'性价比':>8s} {'时间到价值'}")
rows = []
for name, hours, comp, pers, t2v in OUTPUTS:
    roi = (SCORE[comp] + SCORE[pers]) / hours * 100
    rows.append((name, roi))
    print(f'{name:<20s} {hours:>6d} {comp:>9s} {pers:>9s} {roi:>8.1f} {t2v}')

rows.sort(key=lambda r: -r[1])
print(f'\n按性价比排序: ' + ' > '.join(n for n, _ in rows))
best = rows[0][0]
assert '内部演示' in best, f'内部演示应性价比最高，实际 {best}'
paper_roi = dict(rows)['顶会论文（一篇）']
talk_roi = dict(rows)['内部演示（一次）']
assert talk_roi > paper_roi * 10
print(f'\n✅ 内部演示的性价比是论文的 {talk_roi/paper_roi:.0f} 倍（8 小时 vs 300 小时），')
print('   而且它**直接决定你的工作是否被采用**。很多人在论文上投入过度、演示上投入不足。')
patent_roi = dict(rows)['专利（一件）']
assert patent_roi > paper_roi
print(f'✅ 专利的性价比也高于论文（{patent_roi:.1f} vs {paper_roi:.1f}），且**不要求达到发表水平** ——')
print('   很多不够发论文的工程改进是可专利的。')
print('⚠️  内部文档价值最高但最不被认可（激励错配）。缓解：把它做成可见的产出。')

## ✏️ 练习 1：公开前的门禁检查

实现 `publication_gate(item)`：`item` 含
`{'kind': 'paper'|'blog'|'open_source'|'talk', 'ip_disclosed': bool,
'ip_cleared': bool, 'content_reviewed': bool, 'license_reviewed': bool,
'contains_customer_data': bool, 'weeks_until_deadline': int}`。
返回 `{'allowed': bool, 'blockers': [...], 'warnings': [...]}`。
规则：未披露或未获 IP 放行 → 阻断；未过内容审查 → 阻断；含客户数据 → 阻断；
`open_source` 且未做许可审查 → 阻断；剩余时间 < 3 周 → 警告（流程通常要 2–8 周）。

In [ ]:
def publication_gate(item):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ready = dict(kind='paper', ip_disclosed=True, ip_cleared=True, content_reviewed=True,
             license_reviewed=True, contains_customer_data=False, weeks_until_deadline=8)
assert publication_gate(ready)['allowed']
no_ip = dict(ready, ip_disclosed=False)
r = publication_gate(no_ip)
assert not r['allowed'] and any('披露' in b for b in r['blockers'])
cust = dict(ready, contains_customer_data=True)
assert not publication_gate(cust)['allowed']
oss = dict(ready, kind='open_source', license_reviewed=False)
r2 = publication_gate(oss)
assert not r2['allowed'] and any('许可' in b for b in r2['blockers'])
rush = dict(ready, weeks_until_deadline=1)
r3 = publication_gate(rush)
assert r3['allowed'] and r3['warnings'], '时间紧应给警告而不是阻断'
for name, it in [('齐备', ready), ('未披露', no_ip), ('含客户数据', cust),
                 ('开源未查许可', oss), ('只剩一周', rush)]:
    g = publication_gate(it)
    print(f'{name:<14s} allowed={str(g["allowed"]):<6s} '
          f'blockers={g["blockers"]} warnings={g["warnings"]}')
print('✅ 练习 1 通过：**内部流程要 2–8 周，投稿截止前一周才启动是来不及的**')

## ✏️ 练习 2：依赖许可清单

实现 `license_manifest(deps, dep_license, usage)`：`usage` 含
`{'closed_source': bool, 'distributed': bool, 'network_service': bool}`。
返回 `{'blockers': [(依赖, 许可, 原因)], 'reviews': [...], 'ok': bool}`——
对每个依赖调用 `can_use`，`False` 进 blockers，`'review'` 进 reviews。

In [ ]:
def license_manifest(deps, dep_license, usage):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
USAGE_SAAS = dict(closed_source=True, distributed=False, network_service=True)
m = license_manifest(DEPS, DEP_LICENSE, USAGE_SAAS)
assert not m['ok']
assert any(d == 'some-agpl-index' for d, _, _ in m['blockers']), m['blockers']
clean = {k: v for k, v in DEP_LICENSE.items() if k != 'some-agpl-index'}
m2 = license_manifest({k: [d for d in v if d != 'some-agpl-index']
                       for k, v in DEPS.items()}, clean, USAGE_SAAS)
assert m2['ok'], m2
lgpl = dict(clean); lgpl['utils'] = 'LGPL-3.0'
m3 = license_manifest(DEPS, lgpl, USAGE_SAAS)
assert any(d == 'utils' for d, _, _ in m3['reviews']), 'LGPL 应进 review 而不是 blocker'
print('SaaS + 含 AGPL:')
for d, l, why in m['blockers']: print(f'   ❌ {d} ({l}): {why}')
print(f'\n清理后: ok={m2["ok"]}')
print(f'含 LGPL: reviews={[(d,l) for d,l,_ in m3["reviews"]]}')
print('✅ 练习 2 通过：**问题常藏在二级依赖里**，要遍历整棵树')

## ✏️ 练习 3：演示改写

实现 `rewrite_titles(academic_titles, findings)`：把学术章节名改写成断言式标题。
`findings` 是 `{章节名: 断言}` 的映射；没有对应断言的章节保留原名并在返回的
第二个元素里报出来。返回 `(新标题列表, 未改写的章节列表)`。

In [ ]:
def rewrite_titles(academic_titles, findings):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
FINDINGS = {
    '背景与动机': '32k 上下文被激活显存卡住，这是当前最大的产品限制',
    '方法': '按层统计量自适应选择重算的层',
    '结果': '显存降 40%，精度损失 0.3 分，训练慢 8%',
    '消融实验': '统计量的选择不敏感；层数阈值是唯一需要调的超参',
}
new, missed = rewrite_titles(ACADEMIC_TEMPLATE, FINDINGS)
assert len(new) == len(ACADEMIC_TEMPLATE)
assert new[0] == FINDINGS['背景与动机']
assert set(missed) == {'相关工作', '实验设置', '结论与未来工作'}, missed
for t in new: print('  ·', t)
print(f'\n未改写: {missed}')
res = check_talk_structure(new, ask='批准 200 GPU 小时', time_split={'method': 0.2})
print(f'改写后的结构检查: {len(res["problems"])} 个问题（原本 '
      f'{len(check_talk_structure(ACADEMIC_TEMPLATE)["problems"])} 个）')
assert len(res['problems']) < len(check_talk_structure(ACADEMIC_TEMPLATE)['problems'])
print('✅ 练习 3 通过：**标题是断言，连起来读成一个完整论证**')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def publication_gate(item):
    blockers, warnings = [], []
    if not item.get('ip_disclosed'):
        blockers.append('未做内部发明披露 -> 公开会永久丧失新颖性')
    elif not item.get('ip_cleared'):
        blockers.append('IP 评估未放行（等法务确认是否先申请专利）')
    if not item.get('content_reviewed'):
        blockers.append('未过内容审查（产品计划/客户信息/受限数据）')
    if item.get('contains_customer_data'):
        blockers.append('含客户数据 -> 合同与隐私风险，必须移除或获授权')
    if item.get('kind') == 'open_source' and not item.get('license_reviewed'):
        blockers.append('开源发布未做依赖许可审查')
    if item.get('weeks_until_deadline', 99) < 3:
        warnings.append(f'距截止仅 {item["weeks_until_deadline"]} 周，'
                        f'而内部流程通常要 2–8 周')
    return {'allowed': not blockers, 'blockers': blockers, 'warnings': warnings}

In [ ]:
# 练习 2 参考答案
def license_manifest(deps, dep_license, usage):
    seen = set()
    def walk(node):
        if node in seen: return
        seen.add(node)
        for d in deps.get(node, []): walk(d)
    for root in deps: walk(root)
    blockers, reviews = [], []
    for d in sorted(seen):
        lic = dep_license.get(d)
        if not lic: continue
        ok, why = can_use(lic, usage['closed_source'], usage['distributed'],
                          usage['network_service'])
        if ok is False: blockers.append((d, lic, why))
        elif ok == 'review': reviews.append((d, lic, why))
    return {'blockers': blockers, 'reviews': reviews, 'ok': not blockers}

In [ ]:
# 练习 3 参考答案
def rewrite_titles(academic_titles, findings):
    new, missed = [], []
    for t in academic_titles:
        if t in findings:
            new.append(findings[t])
        else:
            new.append(t); missed.append(t)
    return new, missed

---
## 🧪 模板胶囊：三份可直接复制的模板

In [ ]:
TEMPLATES = r'''
════════════════════════════════════════════════════════════════════
① 发明披露书模板（交给专利代理人的东西）
════════════════════════════════════════════════════════════════════
标题：
发明人（全部，按贡献）：
构思日期：____-__-__        ← **决定优先权，务必准确**
对外公开状态：□ 从未 □ 已通过 ______ 公开（若已公开，**立刻联系法务**）
代码位置：

① 技术问题
   现有技术在 ______ 场景下无法 ______，具体瓶颈是 ______。
   （写「技术瓶颈」，不要写「效果不够好」）

② 最接近的现有技术（诚实列出，隐瞒会在审查时反噬）
   · 方案 A：______ ，不足：______
   · 方案 B：______ ，不足：______

③ 本发明的方案（核心机制 + 伪代码/框图）

④ 为什么有效
   机制解释：______
   实验证据：______
   ⭐ **非预期的效果**：______        ← 非显而易见性的最有力论据

⑤ 变体（列尽所有能想到的替代实现 —— 最易跳过、最有价值）
   · 不同架构 / 不同精度 / 不同粒度 / 不同层 / 连续 vs 离散 …
   自问：「竞争者绕过我的最小改动是什么？」把答案也写进来。

⑥ 技术效果的量化（绑定具体技术效果，这决定可专利主题）
   ______ 降低 __%，在 ______ 条件下测得。

════════════════════════════════════════════════════════════════════
② 内部技术演示模板（6 张，每张标题都是断言）
════════════════════════════════════════════════════════════════════
1. 结论：<具体数字的断言>                        ← 第一张就给
2. 为什么值得做：<解锁了什么产品能力>
3. 关键证据：<多少模型 × 多少数据集上复现>
4. 代价与风险：<训练慢多少 / 需重调什么 / 什么情况下失效>   ← 必须有
5. 与现有方案对比：<只保留影响决策的比较>
6. ask：<谁 在 什么时候 做 什么决定 / 需要什么资源>          ← 必须具体

自检：只看第 1 张和第 6 张，一个人能做出决定吗？
自检：把 6 个标题连起来读，是一个完整论证吗？

════════════════════════════════════════════════════════════════════
③ 依赖许可清单（在**引入依赖时**填，不是发布前）
════════════════════════════════════════════════════════════════════
| 依赖/数据集/模型 | 版本 | 许可 | 用途 | 是否分发 | 是否网络服务 | 结论 |
|---|---|---|---|---|---|---|
| onnxruntime | 1.18 | MIT | 推理 | 否 | 是 | ✅ |
| <某向量库>  | ...  | MIT（但二级依赖含 AGPL！）| ... | 否 | 是 | ❌ 交法务 |
| <某数据集>  | ...  | CC-BY-NC | 训练 | — | — | ❌ 禁商用 |

规则一：许可检查在引入依赖时做（发布前发现要重写已写好的代码）
规则二：**不要自己判断边界情形**（静态链接、NC 数据训练的模型）—— 交法务
你的职责：准确记录「用了什么、什么许可、怎么用的」
'''
print(TEMPLATES)
for k in ['构思日期', '非预期的效果', '变体', 'ask', '二级依赖', 'CC-BY-NC']:
    assert k in TEMPLATES, k
print('✅ 三份模板覆盖：发明披露 / 内部演示 / 依赖许可清单')

### 小结
- **公开即丧失新颖性**，且多数辖区不可逆。正确顺序：**有想法 → 记录（带日期）→ 内部披露 → 法务确认 → 再决定公开**。
  「先发 arXiv 占坑再申请专利」不是可行策略。
- **可专利主题的关键是把「算法创新」翻译成「技术问题 + 技术手段 + 技术效果」**。纯算法难，绑定具体技术效果容易。
- **「非预期的效果」是非显而易见性最有力的论据** —— 留意实验里那些反直觉的好结果。
- **权利要求过窄 = 竞争者稍改就绕过**。你（发明人）不可替代的贡献是**列尽所有变体**；
  自问「绕过我的最小改动是什么」。
- **AGPL 打破了「不分发就没有义务」的直觉**：网络提供服务即触发 —— 这正是所有 AI 服务的形态。
  **大量数据集是 CC-BY-NC**（禁商用）。问题常藏在**二级依赖**里。
- **许可检查在引入依赖时做，不是发布前**；**边界情形不要自己判断**，你的职责是准确记录。
- **内部演示 ≠ 学术报告**：结论先行、标题是断言、必须讲代价与风险、必须有具体的 ask。
  成功的定义是「产生了一个决定」，不是「讲清楚了」。
- **投入产出**：内部演示性价比是论文的数十倍；专利不要求达到发表水平；内部文档价值最高但最不被认可。
- **一句话**：研究的价值 ≠ 研究的产出。中间隔着一次「翻译成组织能消费的形式」的转换，
  而这个转换的成本远低于研究本身 —— 这就是它值得刻意练习的原因。

**C52 完结。** 至此这门课补齐了 JD 里的最后一批要求：TF/Keras 心智模型、跨框架迁移、
模型导出与运行时、真机 GPU 工作流、以及研究产出与知识产权。